# 2025-10-31: Create CLR/Frequency input files
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology

**Main aim**: In this notebook, I use the `create-frequency-table.py` function to convert my single cell data into frequency files at different levels of cell type resolution. These outputs will be used for any downstream frequency analysis. 

> Note: The main column used for all multiple myeloma analysis is the column `aifi_plot_l3`. This has L3 cell labels abbreviated to fit plotting needs. 

In [1]:
import concurrent.futures
import os
import pandas as pd
from tqdm import tqdm
import scanpy as sc
if not os.path.exists('../../../data/rna/pseudobulk/outputs'):
    os.makedirs('../../../data/rna/pseudobulk/outputs')


path = '../../../data/rna/'

In [2]:
%run ../../00-utilities/functions/python/create_frequency_table.py

## 1. BMMC Frequencies

In [3]:
adata = sc.read_h5ad(path+"final-objects/final-bmmc-raw.h5ad", backed = 'r')

In [4]:
adata = adata[~adata.obs["aifi_plot_l3"].isin(["Plasma (Malignant)", "Plasma (Non-malignant)"])]

In [5]:
print('Miniumum value:', adata.X.min())
print('Maximum value:', adata.X.max())

Miniumum value: 0
Maximum value: 74012


In [6]:
groupby_sample_col = "sample.sampleKitGuid"
visit_col = "label.visitDetails"

visit_order = ["PreTx", "EI", "ASCT90d", "ASCT1y", "ASCT2y", "Healthy"]

desired_cols = [
    "sample.sampleKitGuid",
    "sample.visitDetails",
    "label.visitDetails",
    "sample.daysSinceFirstVisit",
    "sample.diseaseStatesRecordedAtVisit",
    "subject.biologicalSex",
    "subject.birthYear",
    "subject.ethnicity",
    "subject.race",
    "subject.subjectGuid",
    "subject.cmv",
    "cohort.cohortGuid",
    "manual.time_stamp",
    "manual.category",
    "manual.flu_response",
]

### Generate freq at L3 for paper related analyses

In [7]:
for level in ['l2', 'l3']:
    cell_type_col = f'aifi_plot_{level}'
    df = process_single_cell_metadata(
        adata=adata,
        groupby_col=groupby_sample_col,
        celltype_col=cell_type_col,
        desired_cols=desired_cols,
        visit_col=visit_col,
        visit_order=visit_order
    )
    df.to_csv(f'../../../data/rna/pseudobulk/outputs/bmmc_{level}_frequency.csv', index=False)

## 2. PBMC Frequencies

In [8]:
adata = sc.read_h5ad(path+"final-objects/final-pbmc-raw.h5ad", backed = 'r')

In [9]:
groupby_sample_col = "sample.sampleKitGuid"
visit_col = "label.visitDetails"

visit_order = ["PreTx", "PI2C", "EI", "ASCT60d", "ASCT1y", "ASCT2y", "Healthy"]

desired_cols = [
    "sample.sampleKitGuid",
    "sample.visitDetails",
    "label.visitDetails",
    "sample.visitName",
    "label.visitName",
    "sample.daysSinceFirstVisit",
    "sample.diseaseStatesRecordedAtVisit",
    "subject.biologicalSex",
    "subject.birthYear",
    "subject.ethnicity",
    "subject.race",
    "subject.subjectGuid",
    "subject.cmv",
    "cohort.cohortGuid",
    "manual.time_stamp",
    "manual.category",
    "manual.flu_response",
]

### Generate freq at L3 for paper related analyses

In [10]:
for level in ['l2', 'l3']:
    cell_type_col = f"aifi_plot_{level}"
    df = process_single_cell_metadata(
        adata=adata,
        groupby_col=groupby_sample_col,
        celltype_col=cell_type_col,
        desired_cols=desired_cols,
        visit_col=visit_col,
        visit_order=visit_order,
    )
    df.to_csv(f"../../../data/rna/pseudobulk/outputs/pbmc_{level}_frequency.csv", index=False)